# Analyse Exploratoire des Données AIS
Profiling automatique avec ydata-profiling pour identifier les meilleures features pour l'entraînement.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Connexion à la base PostgreSQL
engine = create_engine('postgresql+psycopg2://postgres:ANjw/84H77C9h2@localhost:5432/ais')

# Lister toutes les tables disponibles
tables = pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'", engine)
print('Tables disponibles :')
print(tables)

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

: 

In [ ]:
# Charger les données - ADAPTER le nom de la table selon le résultat ci-dessus
df = pd.read_sql('SELECT * FROM ais_data LIMIT 50000', engine)
print(f'Shape: {df.shape}')
print(f'Colonnes: {list(df.columns)}')
df.head()

In [ ]:
# Aperçu rapide des types et valeurs manquantes
print(df.dtypes)
print('\n--- Valeurs manquantes ---')
print(df.isnull().sum())
print('\n--- Stats descriptives ---')
df.describe()

In [ ]:
from ydata_profiling import ProfileReport

# Générer le rapport de profiling complet
report = ProfileReport(
    df,
    title='AIS Data Profiling',
    explorative=True,
    correlations={
        'pearson': {'calculate': True},
        'spearman': {'calculate': True},
        'phi_k': {'calculate': True}
    }
)

# Afficher dans le notebook
report.to_notebook_iframe()

In [ ]:
# Sauvegarder le rapport en HTML pour le partager
report.to_file('eda_report_ais.html')
print('Rapport sauvegardé : eda_report_ais.html')

In [ ]:
# Matrice de corrélation pour identifier les meilleures features
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = df.select_dtypes(include='number')
if len(numeric_cols.columns) > 1:
    plt.figure(figsize=(14, 10))
    sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Matrice de corrélation')
    plt.tight_layout()
    plt.show()